# SIFLOW · run_6 · Dream-7B train + eval (SIFLOW-D)

Trains a head-only SIFLOW-D student on the Dream-7B backbone (live, reduced top-m support) and evaluates it. Fills the SIFLOW-D rows of Table 2.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_5 (dream_256.npy + Dream weights cached)

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

In [ ]:
!python scripts/train.py --config siflow/config/dream.yaml --set \
    data.tokens_path={BASE}/data/dream_256.npy \
    out_dir={BASE}/ckpt/dream run_id=siflow_dream train.total_steps=15000

In [ ]:
!python scripts/evaluate.py --config siflow/config/dream.yaml --system siflow \
    --ckpt-dir {BASE}/ckpt/dream --ref-tokens {BASE}/data/dream_val.npy \
    --n-samples 500 --k-list 1 4 --out results/dream.json

In [ ]:
!python scripts/make_tables.py --results results

In [ ]:
# --- Save outputs to Drive (so the next notebook can resume) ---
!cp -r results {BASE}/ 2>/dev/null || true
!cp -r paper/tables_auto.tex {BASE}/ 2>/dev/null || true
print('saved to', BASE)